# 手写数字识别实验入口

这个 notebook 用于分阶段运行：参数检查、可选数据准备、HPO、clean 训练、robust 微调、模型评估、ensemble 权重搜索、文件夹预测与 preprocess debug。默认只做评估，不会自动训练。

In [ ]:
from pathlib import Path
from datetime import datetime
import importlib
import json
import sys
import time

import pandas as pd
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path(r"E:\\ALL\\学习\\AI导论作业-识别手写数字")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.download_finetune_data as download_finetune_data
import src.ensemble_predict as ensemble_predict
import src.evaluate as evaluate_module
import src.hpo as hpo_module
import src.robust_train as robust_train
import src.validation_board as validation_board

for module in [download_finetune_data, ensemble_predict, evaluate_module, hpo_module, robust_train, validation_board]:
    importlib.reload(module)

from src.config import ExperimentConfig, ensure_project_paths
from src.data import create_dataloaders
from src.download_finetune_data import prepare_all as prepare_finetune_datasets
from src.engine import fit
from src.ensemble_predict import predict_image_folder, search_ensemble_weight, predict_batch_pair
from src.evaluate import evaluate_external_holdouts, evaluate_mnist_c_zip, load_model_from_checkpoint
from src.hpo import run_hpo
from src.model import build_model, count_model_parameters
from src.predict import PredictionImageDataset, save_preprocess_debug_visualization, write_predictions_csv
from src.robust_train import run_robust_finetune
from src.train import set_seed
from src.validation_board import evaluate_validation_board

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
{"project_root": str(PROJECT_ROOT), "device": DEVICE, "cuda_available": torch.cuda.is_available()}

## 1. 运行开关与可调参数

默认只评估已有 checkpoint。需要训练、调参、预测时，只打开对应开关。

In [ ]:
DO_PREPARE_DATA = False
DO_HPO = False
DO_TRAIN_CLEAN = False
DO_TRAIN_ROBUST = True
DO_EVALUATE = True
DO_ENSEMBLE_SEARCH = True
DO_PREDICT_FOLDER = False

PATH_CONFIG = {
    "output_dir": PROJECT_ROOT / "outputs_submission",
    "exam_image_dir": PROJECT_ROOT / "exam_data" / "test",
}

CHECKPOINT_CANDIDATES = [
    PATH_CONFIG["output_dir"] / "checkpoints" / "best_model_stat-09987e.pt",
    PATH_CONFIG["output_dir"] / "checkpoints" / "checkpoint_clean_best.pth",
    PATH_CONFIG["output_dir"] / "checkpoints" / "best_model_state_09974.pt",
    PATH_CONFIG["output_dir"] / "checkpoints" / "best_model_state.pt",
]
CLEAN_CHECKPOINT = next((path for path in CHECKPOINT_CANDIDATES if path.exists()), CHECKPOINT_CANDIDATES[-1])
ROBUST_CHECKPOINT = PATH_CONFIG["output_dir"] / "checkpoints" / "robust_expert_best.pt"

DATA_CONFIG = {
    "dataset_name": "multisource",
    "use_mnist": True,
    "use_emnist_digits": True,
    "use_usps": True,
    "use_qmnist": True,
    "emnist_max_samples": 50000,
    "qmnist_max_samples": 60000,
    "external_holdout_names": ("mnist_test", "emnist_digits_test", "qmnist_test10k"),
}
MODEL_CONFIG = {"model_name": "medium_cnn", "dropout": 0.21672530847241062}
TRAIN_CONFIG = {
    "batch_size": 512,
    "epochs": 60,
    "learning_rate": 0.0008398721379146775,
    "optimizer_type": "AdamW",
    "scheduler_type": "CosineAnnealingLR",
    "weight_decay": 6.602542933207749e-06,
    "label_smoothing": 0.03,
    "num_workers": 0,
    "pin_memory": False,
    "persistent_workers": False,
    "prefetch_factor": None,
    "dataloader_timeout": 0,
    "use_amp": True,
    "allow_tf32": True,
    "use_early_stopping": True,
    "early_stopping_patience": 7,
}
AUGMENT_CONFIG = {
    "rotation_degrees": 7.536974266650085,
    "translate_ratio": 0.05567431908414762,
    "scale_min": 0.9213414099692713,
    "scale_max": 1.1010803603468335,
    "shear_degrees": 4.9755817645420075,
    "use_random_affine": True,
    "use_gaussian_blur": False,
}
ROBUST_CONFIG = {
    "fine_tune_lr": 5e-5,
    "fine_tune_epochs": 12,
    "batch_size": 128,
    "robust_aug_strength": "medium",
    "mnist_family_weight": 0.55,
    "use_hasyv2": True,
    "hasyv2_dir": PROJECT_ROOT / "data" / "hasyv2_digits",
    "hasyv2_weight": 0.16,
    "use_chars74k": True,
    "chars74k_dir": PROJECT_ROOT / "data" / "chars74k_digits",
    "chars74k_weight": 0.07,
    "use_penbased_rendered": True,
    "penbased_dir": PROJECT_ROOT / "data" / "penbased_rendered",
    "penbased_weight": 0.12,
}
EVAL_CONFIG = {"external_validation_batch_size": 512, "enable_validation_board": True}
ENSEMBLE_CONFIG = {
    "ensemble_weight_clean": 0.60,
    "ensemble_weight_grid": (0.80, 0.75, 0.70, 0.65, 0.60, 0.55, 0.50, 0.45, 0.40, 0.35, 0.30),
    "use_tta": True,
    "tta_n": 8,
}
DEBUG_CONFIG = {"verbose": True, "log_interval": 50, "debug_preprocess": True, "debug_preprocess_samples": 16}

base_config = ExperimentConfig(
    project_root=PROJECT_ROOT,
    output_dir=PATH_CONFIG["output_dir"],
    clean_checkpoint_path=CLEAN_CHECKPOINT,
    robust_checkpoint_path=ROBUST_CHECKPOINT,
    seed=42,
    **DATA_CONFIG, **MODEL_CONFIG, **TRAIN_CONFIG, **AUGMENT_CONFIG, **EVAL_CONFIG, **ENSEMBLE_CONFIG, **DEBUG_CONFIG,
)

clean_config = base_config
robust_config = ExperimentConfig(
    **{**base_config.to_dict(), **ROBUST_CONFIG,
       "training_mode": "robust_finetune",
       "clean_checkpoint_path": str(CLEAN_CHECKPOINT),
       "checkpoint_name": "robust_expert_best.pt",
       "learning_rate": ROBUST_CONFIG["fine_tune_lr"],
       "weight_decay": 1e-5,
       "freeze_backbone_first": False,
       "use_local_digits": False,
       "local_digits_weight": 0.0,
       "use_optical_digits": False,
       "optical_weight": 0.0}
)

paths = ensure_project_paths(clean_config)
set_seed(clean_config.seed)
run_results = {"run_info": {}, "stages": [], "training": {}, "validation_board": {}, "holdouts": {}, "ensemble": {}, "prediction": {}, "debug_preprocess": {}}


## 2. Run Card

每个 stage 前都会打印本次运行关键信息。

In [ ]:
def stage_card(stage, config, checkpoint=None, datasets=None):
    card = {
        "stage": stage,
        "checkpoint": str(checkpoint) if checkpoint else "",
        "datasets": datasets or [],
        "batch_size": config.batch_size,
        "learning_rate": config.learning_rate,
        "epochs": config.epochs,
        "use_tta": config.use_tta,
        "tta_n": config.tta_n,
        "robust_aug_strength": config.robust_aug_strength,
        "output_dir": str(config.resolved_output_dir()),
        "device": DEVICE,
    }
    print("=" * 80)
    print(json.dumps(card, indent=2, ensure_ascii=False))
    return card

def record_stage(stage, status="done", metric=None, artifact=None, notes=None, start_time=None):
    row = {
        "stage": stage,
        "status": status,
        "metric": metric,
        "artifact": str(artifact) if artifact else "",
        "notes": notes or "",
        "duration_sec": round(time.perf_counter() - start_time, 2) if start_time else None,
    }
    run_results["stages"].append(row)
    return row

run_results["run_info"] = {
    "time": datetime.now().isoformat(timespec="seconds"),
    "project_root": str(PROJECT_ROOT),
    "device": DEVICE,
    "clean_checkpoint": str(CLEAN_CHECKPOINT),
    "robust_checkpoint": str(ROBUST_CHECKPOINT),
}
pd.DataFrame([run_results["run_info"]])

## 3. 可选：数据准备

下载数据通常只需要第一次运行；默认关闭。

In [ ]:
if DO_PREPARE_DATA:
    start = time.perf_counter()
    stage_card("prepare_data", clean_config, datasets=["HASYv2", "Chars74K", "Pen-Based"])
    manifest = prepare_finetune_datasets(PROJECT_ROOT, force=False)
    run_results["datasets"] = manifest
    record_stage("prepare_data", artifact=PROJECT_ROOT / "data" / "finetune_datasets_manifest.json", notes=json.dumps(manifest, ensure_ascii=False), start_time=start)
    manifest
else:
    print("Skipped data preparation.")

## 4. 单独 HPO / 调参

In [ ]:
if DO_HPO:
    start = time.perf_counter()
    stage_card("hpo", clean_config, datasets=["MNIST-family small sample"])
    hpo_result = run_hpo(clean_config, n_trials=12, trial_epochs=5, trial_max_samples=12000, device=DEVICE)
    run_results["hpo"] = hpo_result
    metric = hpo_result["best"]["best_val_accuracy"] if hpo_result.get("best") else None
    record_stage("hpo", metric=metric, artifact=hpo_result.get("hpo_dir"), start_time=start)
    pd.DataFrame(hpo_result["rows"])
else:
    print("Skipped HPO.")

## 5. 单独训练 Clean Expert

In [ ]:
if DO_TRAIN_CLEAN:
    start = time.perf_counter()
    stage_card("train_clean", clean_config, datasets=["MNIST", "QMNIST", "EMNIST digits", "USPS"])
    train_loader, val_loader = create_dataloaders(clean_config)
    clean_model = build_model(clean_config).to(DEVICE)
    total_params, trainable_params = count_model_parameters(clean_model)
    history = fit(clean_model, train_loader, val_loader, config=clean_config, paths=paths, device=DEVICE)
    checkpoint = paths.checkpoints_dir / clean_config.checkpoint_name
    run_results["training"]["clean"] = history
    record_stage("train_clean", metric=history.get("best_val_accuracy"), artifact=checkpoint, notes=f"params={total_params}, trainable={trainable_params}", start_time=start)
    history
else:
    print("Skipped clean training.")

## 6. 单独训练 Robust Expert

In [ ]:
if DO_TRAIN_ROBUST:
    start = time.perf_counter()
    stage_card("train_robust", robust_config, checkpoint=CLEAN_CHECKPOINT, datasets=["MNIST-family", "HASYv2", "Chars74K", "Pen-Based"])
    history = run_robust_finetune(robust_config)
    run_results["training"]["robust"] = history
    metric = history["full_finetune"].get("best_val_accuracy")
    record_stage("train_robust", metric=metric, artifact=ROBUST_CHECKPOINT, start_time=start)
    history
else:
    print("Skipped robust fine-tuning.")

## 7. 单独评估模型

默认评估 clean checkpoint；如果 robust checkpoint 存在，也会一起评估 robust。

In [ ]:
def evaluate_named_checkpoint(name, checkpoint, config):
    checkpoint = Path(checkpoint)
    if not checkpoint.exists():
        row = record_stage(f"evaluate_{name}", status="missing_checkpoint", artifact=checkpoint)
        return row
    start = time.perf_counter()
    stage_card(f"evaluate_{name}", config, checkpoint=checkpoint, datasets=list(config.external_holdout_names))
    model, payload = load_model_from_checkpoint(checkpoint, config, DEVICE)
    board = evaluate_validation_board(model, config, paths.logs_dir, DEVICE, prefix=name)
    holdouts = evaluate_external_holdouts(model, config=config, output_dir=paths.evaluation_dir / f"holdouts_{name}", device=DEVICE)
    run_results["validation_board"][name] = board
    run_results["holdouts"][name] = holdouts
    metric = board["score"]["composite_score"]
    return record_stage(f"evaluate_{name}", metric=metric, artifact=paths.logs_dir / f"validation_board_{name}.json", start_time=start)

if DO_EVALUATE:
    eval_rows = []
    eval_rows.append(evaluate_named_checkpoint("clean", CLEAN_CHECKPOINT, clean_config))
    if ROBUST_CHECKPOINT.exists():
        eval_rows.append(evaluate_named_checkpoint("robust", ROBUST_CHECKPOINT, robust_config))
    pd.DataFrame(eval_rows)
else:
    print("Skipped evaluation.")

## 8. 单独 Ensemble 权重搜索

In [ ]:
if DO_ENSEMBLE_SEARCH:
    if not ROBUST_CHECKPOINT.exists():
        raise FileNotFoundError(f"robust checkpoint 不存在: {ROBUST_CHECKPOINT}")
    start = time.perf_counter()
    stage_card("ensemble_search", robust_config, checkpoint=f"{CLEAN_CHECKPOINT} + {ROBUST_CHECKPOINT}", datasets=["validation board"])
    clean_model, _ = load_model_from_checkpoint(CLEAN_CHECKPOINT, clean_config, DEVICE)
    robust_model, _ = load_model_from_checkpoint(ROBUST_CHECKPOINT, robust_config, DEVICE)
    best, rows = search_ensemble_weight(clean_model, robust_model, robust_config, DEVICE, paths.logs_dir)
    run_results["ensemble"] = {"best": best, "rows": rows}
    record_stage("ensemble_search", metric=best.get("composite_score") if best else None, artifact=paths.logs_dir / "ensemble_weight_search.csv", notes=json.dumps(best, ensure_ascii=False), start_time=start)
    pd.DataFrame(rows)
else:
    print("Skipped ensemble search.")

## 9. 单独预测 / 文件夹测试 + preprocess debug

In [ ]:
PREDICT_IMAGE_DIR = PATH_CONFIG["exam_image_dir"]
PREDICT_DEBUG_DIR = paths.outputs_dir / "debug_preprocess"

if DO_PREDICT_FOLDER:
    if not ROBUST_CHECKPOINT.exists():
        raise FileNotFoundError(f"robust checkpoint 不存在: {ROBUST_CHECKPOINT}")
    if not PREDICT_IMAGE_DIR.exists():
        raise FileNotFoundError(f"预测图片目录不存在: {PREDICT_IMAGE_DIR}")
    start = time.perf_counter()
    stage_card("predict_folder", robust_config, checkpoint=f"{CLEAN_CHECKPOINT} + {ROBUST_CHECKPOINT}", datasets=[str(PREDICT_IMAGE_DIR)])
    clean_model, _ = load_model_from_checkpoint(CLEAN_CHECKPOINT, clean_config, DEVICE)
    robust_model, _ = load_model_from_checkpoint(ROBUST_CHECKPOINT, robust_config, DEVICE)
    dataset = PredictionImageDataset(PREDICT_IMAGE_DIR, image_size=robust_config.image_size, auto_invert=robust_config.auto_invert)
    loader = DataLoader(dataset, batch_size=robust_config.batch_size, shuffle=False, num_workers=0)
    clean_rows, robust_rows, ensemble_rows = predict_image_folder(clean_model, robust_model, loader, robust_config, DEVICE, clean_weight=robust_config.ensemble_weight_clean)
    write_predictions_csv(clean_rows, paths.predictions_dir / "clean_predictions.csv")
    write_predictions_csv(robust_rows, paths.predictions_dir / "robust_predictions.csv")
    write_predictions_csv(ensemble_rows, paths.predictions_dir / "ensemble_predictions.csv")
    write_predictions_csv(ensemble_rows, paths.outputs_dir / "submission.csv")
    saved_debug = []
    if robust_config.debug_preprocess:
        debug_loader = DataLoader(dataset, batch_size=min(robust_config.debug_preprocess_samples, robust_config.batch_size), shuffle=False, num_workers=0)
        with torch.no_grad():
            for images, filenames in debug_loader:
                clean_prob, robust_prob, ensemble_prob = predict_batch_pair(clean_model, robust_model, images, robust_config, DEVICE, clean_weight=robust_config.ensemble_weight_clean)
                confidence_values, prediction_values = ensemble_prob.max(dim=1)
                for filename, prediction, confidence in zip(filenames, prediction_values.cpu().tolist(), confidence_values.cpu().tolist()):
                    if len(saved_debug) >= robust_config.debug_preprocess_samples:
                        break
                    saved_debug.append(str(save_preprocess_debug_visualization(PREDICT_IMAGE_DIR / filename, PREDICT_DEBUG_DIR, prediction=int(prediction), confidence=float(confidence), image_size=robust_config.image_size, auto_invert=robust_config.auto_invert)))
                break
    run_results["prediction"] = {"submission": str(paths.outputs_dir / "submission.csv"), "num_images": len(ensemble_rows)}
    run_results["debug_preprocess"] = {"dir": str(PREDICT_DEBUG_DIR), "files": saved_debug}
    record_stage("predict_folder", metric=len(ensemble_rows), artifact=paths.outputs_dir / "submission.csv", start_time=start)
    pd.DataFrame(ensemble_rows, columns=["filename", "prediction"]).head()
else:
    print("Skipped folder prediction.")

## 10. 本轮运行结果总览

In [ ]:
summary_df = pd.DataFrame(run_results["stages"])
if not summary_df.empty:
    summary_path = paths.logs_dir / "notebook_run_results.csv"
    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
    print(f"summary saved: {summary_path}")
summary_df